# MSADS 599 - Modeling Part 1 - Classification - XGBoost

## Setup

Import Packages

In [1]:
!pip install xgboost

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from huggingface_hub import hf_hub_download, login
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# This shows all columns when analyzing the data
pd.set_option('display.max_columns', None)

Connect to HuggingFace and Import Data

In [3]:
# Connection to HuggingFace via API token
login(token="HF_TOKEN")

In [4]:
# Import training data
train_path = hf_hub_download(
    repo_id = "ADS599-Capstone/modeling_data",
    filename = "part1_train/part1_train-00000-of-00001.parquet",
    repo_type = "dataset"
)

train_df = pd.read_parquet(train_path)
print(train_df.shape)

part1_train/part1_train-00000-of-00001.p(…):   0%|          | 0.00/1.32M [00:00<?, ?B/s]

(67085, 73)


In [28]:
# Import testing data
test_path = hf_hub_download(
    repo_id = "ADS599-Capstone/modeling_data",
    filename = "part1_test/part1_test-00000-of-00001.parquet",
    repo_type = "dataset"
)

test_df = pd.read_parquet(test_path)
print(test_df.shape)

(16882, 73)


## Modeling

Clean and Set the Target Variable

In [34]:
icu_labels = {'ED_DIRECT_ICU', 'ED_WARD_ICU'}

train_df['target'] = train_df['cohort_label'].apply(lambda x: 1 if x in icu_labels else 0)
test_df['target'] = test_df['cohort_label'].apply(lambda x: 1 if x in icu_labels else 0)

# Fit ONLY on train, transform both
le = LabelEncoder()
train_df['target'] = le.fit_transform(train_df['target'])
test_df['target'] = le.transform(test_df['target'])  # ← transform only, not fit_transform

class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(class_mapping)

{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}


In [36]:
# Split the target variable from the test and train datasets

# Training Set
X_train = train_df.drop(columns=['cohort_label', 'target'])
y_train = train_df['target']

# Testing Set
X_test = test_df.drop(columns=['cohort_label', 'target'])
y_test = test_df['target']

# Sanity Check
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (67085, 72), y_train: (67085,)
X_test: (16882, 72), y_test: (16882,)


Train the Model

In [37]:
print(y_train.unique())
print(y_test.unique())

[0 1]
[0 1]


In [38]:
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',         # Multiclass log loss
    n_estimators=300,
    learning_rate=0.1,
    max_depth=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=35
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

[0]	validation_0-logloss:0.49511
[50]	validation_0-logloss:0.37008
[100]	validation_0-logloss:0.38627
[150]	validation_0-logloss:0.40463
[200]	validation_0-logloss:0.42065
[250]	validation_0-logloss:0.43467
[299]	validation_0-logloss:0.44697


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=20, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [39]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'max_depth':        [4, 6, 10, 20],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.3],
}

xgb_base = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=35,
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=30,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=35,
    verbose=2,
)

xgb_search.fit(X_train, y_train)
print("Best XGB params:", xgb_search.best_params_)
print("Best XGB score:", xgb_search.best_score_)

xgb_model = xgb_search.best_estimator_  # Drop-in replacement for your original xgb_model

# Best XGBoost Hyperparameters
# {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}

Fitting 5 folds for each of 30 candidates, totalling 150 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:27:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best XGB params: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.6}
Best XGB score: 0.8306818932923694


Evaluate the Model

In [40]:
target_names = ['Discharge', 'ICU']  # index 0=Discharge, 1=ICU

y_pred = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

TypeError: object of type 'numpy.int64' has no len()

Feature Importance - XGBoost

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(xgb_model, max_num_features=20, ax=ax)
plt.title("Top 20 Most Important Features")
plt.tight_layout()
plt.show()